# Script 4 Final EEG / ECG / POMS Analysis

Final analysis for the 20 included participants after excluding **IC12** and **N07**.

This notebook:
1. Reads the frozen Script 3 scene-level EEG output.
2. Extracts a simple mean heart-rate estimate from CH3 in the original recording files.
3. Reads the Pre/Post POMS subscale totals and calculates TMD.
4. Runs conservative within-subject non-parametric statistics.
5. Produces compact publication-ready figures and export tables.

Primary EEG outcomes: bilateral relative theta, alpha and beta power.  
ECG is used only as an auxiliary mean-heart-rate measure.  
POMS is analysed at participant level (Pre vs Post), not repeated across scenes.

In [1]:
# =========================================================
# 0. IMPORTS
# =========================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal, stats

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Imports complete.")

Imports complete.


In [2]:
# =========================================================
# 1. USER SETTINGS
# =========================================================

# Same root folder used in Script 3.
DATA_DIR = Path(r"C:\Users\Harvey McWalter\Desktop\EEG_Data_22")

SCRIPT3_DIR = DATA_DIR / "Script3_EEG_Results"

# Script 3 intentionally exported this compact file for Script 4.
SCRIPT3_INPUT_FILE = SCRIPT3_DIR / "script3_for_script4.csv"

# Fallback if the compact file is not present.
SCRIPT3_FALLBACK_FILE = SCRIPT3_DIR / "script3_scene_eeg_summary.csv"

# POMS input file.
POMS_FILE = DATA_DIR / "POMS Data.xlsx"

OUTPUT_DIR = DATA_DIR / "Script4_Final_Analysis"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_PARTICIPANTS = (
    [f"IC{i:02d}" for i in range(1, 16)]
    + [f"N{i:02d}" for i in range(1, 8)]
)

EXCLUDED_PARTICIPANTS = {"IC12", "N07"}
INCLUDED_PARTICIPANTS = [
    p for p in EXPECTED_PARTICIPANTS
    if p not in EXCLUDED_PARTICIPANTS
]

SCENE_ORDER = ["BG", "G", "GR"]
SCENE_NAMES = {
    "BG": "Blue-Green",
    "G": "Green",
    "GR": "Grey",
}

# Raw recording settings.
FS = 500
ECG_COL = "CH3"
START_TRIM_SECONDS = 15.0

# Simple ECG settings.
ECG_BAND = (5.0, 20.0)
ECG_FILTER_ORDER = 3
MIN_RR_SECONDS = 0.40
MAX_RR_SECONDS = 1.50

PLOT_DPI = 300

print("Data folder:", DATA_DIR)
print("Included participants:", len(INCLUDED_PARTICIPANTS))
print("Excluded participants:", sorted(EXCLUDED_PARTICIPANTS))

Data folder: C:\Users\Harvey McWalter\Desktop\EEG_Data_22
Included participants: 20
Excluded participants: ['IC12', 'N07']


In [3]:
# =========================================================
# 2. GENERAL HELPERS
# =========================================================

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted = np.empty(m, dtype=float)

    running_max = 0.0
    for rank, index in enumerate(order):
        candidate = (m - rank) * p[index]
        running_max = max(running_max, candidate)
        adjusted[index] = min(running_max, 1.0)

    return adjusted


def rank_biserial_paired(x, y):
    # Positive value means x tends to be greater than y.
    d = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    d = d[np.isfinite(d) & (d != 0)]

    if len(d) == 0:
        return 0.0

    ranks = stats.rankdata(np.abs(d), method="average")
    w_plus = ranks[d > 0].sum()
    w_minus = ranks[d < 0].sum()
    denominator = w_plus + w_minus

    return float((w_plus - w_minus) / denominator) if denominator > 0 else 0.0


def median_iqr(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    return {
        "n": len(values),
        "median": float(np.median(values)),
        "q1": float(np.percentile(values, 25)),
        "q3": float(np.percentile(values, 75)),
        "iqr": float(np.percentile(values, 75) - np.percentile(values, 25)),
    }


def run_three_scene_nonparametric(
    table,
    value_column,
    subject_column="participant",
    condition_column="scene_code",
):
    pivot = (
        table.pivot(
            index=subject_column,
            columns=condition_column,
            values=value_column,
        )
        .reindex(columns=SCENE_ORDER)
        .dropna()
    )

    if len(pivot) < 3:
        raise ValueError(
            f"{value_column}: fewer than 3 complete participants."
        )

    arrays = [pivot[scene].to_numpy(dtype=float) for scene in SCENE_ORDER]

    friedman = stats.friedmanchisquare(*arrays)
    n = len(pivot)
    k = len(SCENE_ORDER)
    kendalls_w = float(friedman.statistic / (n * (k - 1)))

    omnibus = {
        "feature": value_column,
        "n_complete": n,
        "friedman_chi2": float(friedman.statistic),
        "df": k - 1,
        "p_value": float(friedman.pvalue),
        "kendalls_w": kendalls_w,
    }

    pair_definitions = [
        ("BG", "G"),
        ("BG", "GR"),
        ("G", "GR"),
    ]

    pairwise_rows = []
    raw_p_values = []

    for scene_a, scene_b in pair_definitions:
        a = pivot[scene_a].to_numpy(dtype=float)
        b = pivot[scene_b].to_numpy(dtype=float)

        if np.allclose(a, b, equal_nan=True):
            statistic = 0.0
            p_value = 1.0
        else:
            test = stats.wilcoxon(
                a,
                b,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto",
            )
            statistic = float(test.statistic)
            p_value = float(test.pvalue)

        raw_p_values.append(p_value)

        pairwise_rows.append({
            "feature": value_column,
            "scene_a": scene_a,
            "scene_b": scene_b,
            "comparison": f"{SCENE_NAMES[scene_a]} vs {SCENE_NAMES[scene_b]}",
            "n_complete": n,
            "wilcoxon_statistic": statistic,
            "p_raw": p_value,
            "rank_biserial_a_minus_b": rank_biserial_paired(a, b),
            "median_difference_a_minus_b": float(np.median(a - b)),
        })

    adjusted = holm_adjust(raw_p_values)

    for row, p_adj in zip(pairwise_rows, adjusted):
        row["p_holm"] = float(p_adj)

    descriptive_rows = []

    for scene in SCENE_ORDER:
        summary = median_iqr(pivot[scene].to_numpy(dtype=float))
        descriptive_rows.append({
            "feature": value_column,
            "scene_code": scene,
            "scene_name": SCENE_NAMES[scene],
            **summary,
        })

    return (
        pd.DataFrame([omnibus]),
        pd.DataFrame(pairwise_rows),
        pd.DataFrame(descriptive_rows),
        pivot,
    )


def save_figure(fig, stem):
    png_path = FIGURE_DIR / f"{stem}.png"
    pdf_path = FIGURE_DIR / f"{stem}.pdf"

    fig.savefig(png_path, dpi=PLOT_DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    return png_path, pdf_path


print("Helper functions ready.")

Helper functions ready.


In [4]:
# =========================================================
# 3. LOAD AND VALIDATE FROZEN SCRIPT 3 EEG RESULTS
# =========================================================

if SCRIPT3_INPUT_FILE.exists():
    eeg = pd.read_csv(SCRIPT3_INPUT_FILE)
    eeg_source = SCRIPT3_INPUT_FILE

elif SCRIPT3_FALLBACK_FILE.exists():
    eeg = pd.read_csv(SCRIPT3_FALLBACK_FILE)
    eeg_source = SCRIPT3_FALLBACK_FILE

    eeg = eeg.rename(columns={
        "bilateral_relative_theta_median_percent": "eeg_theta_relative_percent",
        "bilateral_relative_alpha_median_percent": "eeg_alpha_relative_percent",
        "bilateral_relative_beta_median_percent": "eeg_beta_relative_percent",
    })

else:
    raise FileNotFoundError(
        "Could not find either Script 3 input file:\n"
        f"  {SCRIPT3_INPUT_FILE}\n"
        f"  {SCRIPT3_FALLBACK_FILE}"
    )

eeg.columns = eeg.columns.astype(str).str.strip()

required_eeg_columns = [
    "participant",
    "scene_code",
    "scene_name",
    "file_name",
    "clean_epochs",
    "usable_duration_s",
    "joint_retention_percent",
    "eeg_theta_relative_percent",
    "eeg_alpha_relative_percent",
    "eeg_beta_relative_percent",
]

missing = [c for c in required_eeg_columns if c not in eeg.columns]
if missing:
    raise KeyError(
        f"Script 3 input is missing required columns: {missing}"
    )

eeg["participant"] = eeg["participant"].astype(str).str.strip().str.upper()
eeg["scene_code"] = eeg["scene_code"].astype(str).str.strip().str.upper()

numeric_eeg_columns = [
    "clean_epochs",
    "usable_duration_s",
    "joint_retention_percent",
    "eeg_theta_relative_percent",
    "eeg_alpha_relative_percent",
    "eeg_beta_relative_percent",
]

for column in numeric_eeg_columns:
    eeg[column] = pd.to_numeric(eeg[column], errors="raise")

if len(eeg) != 66:
    raise ValueError(f"Expected 66 Script 3 scene rows; found {len(eeg)}.")

if eeg["participant"].nunique() != 22:
    raise ValueError(
        f"Expected 22 Script 3 participants; found {eeg['participant'].nunique()}."
    )

duplicate = eeg.duplicated(["participant", "scene_code"], keep=False)
if duplicate.any():
    display(eeg.loc[duplicate])
    raise ValueError("Duplicate participant-scene rows in Script 3 input.")

eeg20 = eeg.loc[
    ~eeg["participant"].isin(EXCLUDED_PARTICIPANTS)
].copy()

if len(eeg20) != 60 or eeg20["participant"].nunique() != 20:
    raise ValueError(
        "After excluding IC12 and N07, expected 60 scene rows "
        "from 20 participants."
    )

expected_keys = {
    (participant, scene)
    for participant in INCLUDED_PARTICIPANTS
    for scene in SCENE_ORDER
}
actual_keys = set(zip(eeg20["participant"], eeg20["scene_code"]))

if actual_keys != expected_keys:
    missing_keys = sorted(expected_keys - actual_keys)
    unexpected_keys = sorted(actual_keys - expected_keys)
    raise ValueError(
        f"Participant-scene mismatch.\n"
        f"Missing: {missing_keys}\n"
        f"Unexpected: {unexpected_keys}"
    )

print("Frozen Script 3 input:", eeg_source)
print("EEG rows before exclusion:", len(eeg))
print("EEG rows after exclusion:", len(eeg20))
print("EEG participants for final analysis:", eeg20["participant"].nunique())
print("Minimum joint retention (%):", round(eeg20["joint_retention_percent"].min(), 1))

display(
    eeg20[
        [
            "participant",
            "scene_code",
            "clean_epochs",
            "joint_retention_percent",
            "eeg_theta_relative_percent",
            "eeg_alpha_relative_percent",
            "eeg_beta_relative_percent",
        ]
    ].head(9)
)

Frozen Script 3 input: C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script3_EEG_Results\script3_for_script4.csv
EEG rows before exclusion: 66
EEG rows after exclusion: 60
EEG participants for final analysis: 20
Minimum joint retention (%): 73.9


,participant,scene_code,clean_epochs,joint_retention_percent,eeg_theta_relative_percent,eeg_alpha_relative_percent,eeg_beta_relative_percent
0,IC01,BG,51,73.913043,66.773248,15.699090,16.160337
1,IC01,G,69,97.183099,87.865620,7.805933,4.426262
2,IC01,GR,71,100.000000,75.154811,12.915692,11.661407
3,IC02,BG,66,95.652174,78.403848,12.441519,9.549123
4,IC02,G,70,100.000000,60.377917,20.803363,17.923323
5,IC02,GR,67,94.366197,59.594660,21.531544,17.738448
6,IC03,BG,69,97.183099,76.578888,13.571728,8.799285
7,IC03,G,70,98.591549,68.125645,20.042685,11.720368
8,IC03,GR,71,100.000000,75.974663,15.719867,8.410298


In [5]:
# =========================================================
# 4. LOAD POMS AND CALCULATE TMD
# =========================================================

if not POMS_FILE.exists():
    raise FileNotFoundError(f"POMS file not found: {POMS_FILE}")

poms = pd.read_excel(POMS_FILE)

# Fixes the current Excel header " Time" -> "Time".
poms.columns = poms.columns.astype(str).str.strip()

required_poms_columns = [
    "Participant",
    "Time",
    "TEN",
    "DEP",
    "ANG",
    "FAT",
    "CON",
    "VIG",
    "ERA",
]

missing = [c for c in required_poms_columns if c not in poms.columns]
if missing:
    raise KeyError(f"POMS file is missing columns: {missing}")

poms["Participant"] = (
    poms["Participant"].astype(str).str.strip().str.upper()
)
poms["Time"] = (
    poms["Time"].astype(str).str.strip().str.title()
)

subscales = ["TEN", "DEP", "ANG", "FAT", "CON", "VIG", "ERA"]

for column in subscales:
    poms[column] = pd.to_numeric(poms[column], errors="raise")

# The spreadsheet already contains subscale totals.
# Individual reverse-scored questionnaire items are therefore not scored again here.
#
# TMD = [TEN + DEP + ANG + FAT + CON] - [VIG + ERA] + 100
poms["TMD"] = (
    poms["TEN"]
    + poms["DEP"]
    + poms["ANG"]
    + poms["FAT"]
    + poms["CON"]
    - poms["VIG"]
    - poms["ERA"]
    + 100
)

if len(poms) != 44 or poms["Participant"].nunique() != 22:
    raise ValueError(
        "Expected 44 POMS rows (22 participants x Pre/Post). "
        f"Found {len(poms)} rows and {poms['Participant'].nunique()} participants."
    )

invalid_time = ~poms["Time"].isin(["Pre", "Post"])
if invalid_time.any():
    display(poms.loc[invalid_time])
    raise ValueError("POMS Time must be Pre or Post.")

counts = (
    poms.groupby(["Participant", "Time"])
    .size()
    .unstack(fill_value=0)
)

if not {"Pre", "Post"}.issubset(counts.columns):
    raise ValueError("POMS must contain both Pre and Post rows.")

if not ((counts["Pre"] == 1) & (counts["Post"] == 1)).all():
    display(counts.loc[~((counts["Pre"] == 1) & (counts["Post"] == 1))])
    raise ValueError("Each participant must have exactly one Pre and one Post row.")

poms20 = poms.loc[
    ~poms["Participant"].isin(EXCLUDED_PARTICIPANTS)
].copy()

poms_wide = (
    poms20.pivot(
        index="Participant",
        columns="Time",
        values=["TEN", "DEP", "ANG", "FAT", "CON", "VIG", "ERA", "TMD"],
    )
)

poms_wide.columns = [
    f"{measure}_{time}"
    for measure, time in poms_wide.columns
]

for measure in [*subscales, "TMD"]:
    poms_wide[f"Delta_{measure}"] = (
        poms_wide[f"{measure}_Post"]
        - poms_wide[f"{measure}_Pre"]
    )

poms_wide = poms_wide.reset_index()

if len(poms_wide) != 20:
    raise ValueError(
        f"Expected 20 included POMS participants; found {len(poms_wide)}."
    )

print("POMS rows before exclusion:", len(poms))
print("POMS participants after exclusion:", len(poms_wide))
print("Definition: Delta_TMD = Post - Pre")
print("Negative Delta_TMD means lower mood disturbance after the session.")

display(
    poms_wide[
        ["Participant", "TMD_Pre", "TMD_Post", "Delta_TMD"]
    ].head(10)
)

POMS rows before exclusion: 44
POMS participants after exclusion: 20
Definition: Delta_TMD = Post - Pre
Negative Delta_TMD means lower mood disturbance after the session.


,Participant,TMD_Pre,TMD_Post,Delta_TMD
0,IC01,88,90,2
1,IC02,77,74,-3
2,IC03,94,91,-3
3,IC04,84,89,5
4,IC05,111,117,6
5,IC06,94,83,-11
6,IC07,93,81,-12
7,IC08,90,89,-1
8,IC09,71,69,-2
9,IC10,130,105,-25


In [6]:
# =========================================================
# 5. SIMPLE ECG PROCESSING
# =========================================================

def parse_raw_filename(file_path):
    pattern = re.compile(
        r"^(?P<participant>[A-Za-z]+\d+)_(?P<scene>BG|GR|G)_EEG$",
        flags=re.IGNORECASE,
    )

    match = pattern.fullmatch(file_path.stem)
    if match is None:
        raise ValueError(file_path.name)

    return (
        match.group("participant").upper(),
        match.group("scene").upper(),
    )


def robust_sd(values):
    values = np.asarray(values, dtype=float)
    median_value = np.median(values)
    mad = np.median(np.abs(values - median_value))
    return 1.4826 * mad


def simple_ecg_heart_rate(file_path):
    # Auxiliary ECG only:
    # CH3 -> trim first 15 s -> 5-20 Hz band-pass -> find QRS peaks -> mean HR.

    participant, scene_code = parse_raw_filename(file_path)

    raw = pd.read_csv(file_path, low_memory=False)
    raw.columns = raw.columns.astype(str).str.strip()

    if ECG_COL not in raw.columns:
        raise KeyError(
            f"{file_path.name}: ECG column '{ECG_COL}' not found. "
            f"Available columns: {raw.columns.tolist()}"
        )

    x = pd.to_numeric(raw[ECG_COL], errors="coerce").to_numpy(dtype=float)

    if not np.all(np.isfinite(x)):
        raise ValueError(f"{file_path.name}: CH3 contains NaN/non-numeric samples.")

    trim_samples = int(round(START_TRIM_SECONDS * FS))
    if len(x) <= trim_samples + FS * 10:
        raise ValueError(f"{file_path.name}: recording is too short after trimming.")

    x = x[trim_samples:]
    x = x - np.median(x)

    sos = signal.butter(
        ECG_FILTER_ORDER,
        ECG_BAND,
        btype="bandpass",
        fs=FS,
        output="sos",
    )
    filtered = signal.sosfiltfilt(sos, x)

    positive_extreme = abs(np.percentile(filtered, 99.5))
    negative_extreme = abs(np.percentile(filtered, 0.5))

    if negative_extreme > positive_extreme:
        detection_signal = -filtered
        detected_polarity = "negative"
    else:
        detection_signal = filtered
        detected_polarity = "positive"

    scale = robust_sd(detection_signal)
    prominence = max(
        3.0 * scale,
        0.05 * np.ptp(detection_signal),
        np.finfo(float).eps,
    )

    peaks, _ = signal.find_peaks(
        detection_signal,
        distance=int(round(MIN_RR_SECONDS * FS)),
        prominence=prominence,
    )

    if len(peaks) < 10:
        prominence = max(
            2.0 * scale,
            0.03 * np.ptp(detection_signal),
            np.finfo(float).eps,
        )
        peaks, _ = signal.find_peaks(
            detection_signal,
            distance=int(round(MIN_RR_SECONDS * FS)),
            prominence=prominence,
        )

    rr_all = np.diff(peaks) / FS

    valid_rr_mask = (
        (rr_all >= MIN_RR_SECONDS)
        & (rr_all <= MAX_RR_SECONDS)
    )
    rr_valid = rr_all[valid_rr_mask]

    duration_s = len(filtered) / FS
    valid_fraction = (
        float(valid_rr_mask.mean())
        if len(valid_rr_mask) > 0
        else 0.0
    )

    if len(rr_valid) >= 10:
        mean_hr_bpm = float(60.0 / np.mean(rr_valid))
        median_hr_bpm = float(60.0 / np.median(rr_valid))
    else:
        mean_hr_bpm = np.nan
        median_hr_bpm = np.nan

    qc_ok = (
        len(rr_valid) >= 20
        and valid_fraction >= 0.80
        and np.isfinite(mean_hr_bpm)
        and 40 <= mean_hr_bpm <= 150
    )

    return {
        "participant": participant,
        "scene_code": scene_code,
        "scene_name": SCENE_NAMES[scene_code],
        "file_name": file_path.name,
        "ecg_duration_s": duration_s,
        "ecg_detected_polarity": detected_polarity,
        "ecg_detected_peaks": len(peaks),
        "ecg_valid_rr_count": len(rr_valid),
        "ecg_valid_rr_fraction": valid_fraction,
        "ecg_mean_hr_bpm": mean_hr_bpm,
        "ecg_median_hr_bpm": median_hr_bpm,
        "ecg_qc": "ok" if qc_ok else "review",
        "ecg_error": "",
    }


raw_files = []

for file_path in DATA_DIR.glob("*_EEG.csv"):
    try:
        participant, scene_code = parse_raw_filename(file_path)

        if (
            participant in INCLUDED_PARTICIPANTS
            and scene_code in SCENE_ORDER
        ):
            raw_files.append(file_path)

    except ValueError:
        pass

raw_files = sorted(raw_files)

if len(raw_files) != 60:
    raise ValueError(
        "Expected 60 included raw files after excluding IC12 and N07; "
        f"found {len(raw_files)}."
    )

ecg_rows = []

for file_path in raw_files:
    try:
        result = simple_ecg_heart_rate(file_path)

    except Exception as error:
        try:
            participant, scene_code = parse_raw_filename(file_path)
        except Exception:
            participant, scene_code = "", ""

        result = {
            "participant": participant,
            "scene_code": scene_code,
            "scene_name": SCENE_NAMES.get(scene_code, ""),
            "file_name": file_path.name,
            "ecg_duration_s": np.nan,
            "ecg_detected_polarity": "",
            "ecg_detected_peaks": 0,
            "ecg_valid_rr_count": 0,
            "ecg_valid_rr_fraction": 0.0,
            "ecg_mean_hr_bpm": np.nan,
            "ecg_median_hr_bpm": np.nan,
            "ecg_qc": "error",
            "ecg_error": str(error),
        }

    ecg_rows.append(result)

ecg = pd.DataFrame(ecg_rows)

print("ECG files processed:", len(ecg))
print("\nECG QC counts:")
print(ecg["ecg_qc"].value_counts(dropna=False))

flagged_ecg = ecg.loc[ecg["ecg_qc"] != "ok"].copy()

if len(flagged_ecg) > 0:
    print("\nECG recordings flagged for review:")
    display(
        flagged_ecg[
            [
                "participant",
                "scene_code",
                "file_name",
                "ecg_mean_hr_bpm",
                "ecg_valid_rr_count",
                "ecg_valid_rr_fraction",
                "ecg_qc",
                "ecg_error",
            ]
        ]
    )
else:
    print("\nAll 60 ECG recordings passed the simple ECG QC.")

ECG files processed: 60

ECG QC counts:
ecg_qc
ok    60
Name: count, dtype: int64

All 60 ECG recordings passed the simple ECG QC.


In [7]:
# =========================================================
# 6. BUILD FINAL SCENE-LEVEL ANALYSIS DATASET
# =========================================================

scene_data = eeg20.merge(
    ecg[
        [
            "participant",
            "scene_code",
            "ecg_mean_hr_bpm",
            "ecg_median_hr_bpm",
            "ecg_valid_rr_count",
            "ecg_valid_rr_fraction",
            "ecg_qc",
        ]
    ],
    on=["participant", "scene_code"],
    how="left",
    validate="one_to_one",
)

scene_data["scene_order"] = scene_data["scene_code"].map({
    "BG": 0,
    "G": 1,
    "GR": 2,
})

scene_data = scene_data.sort_values(
    ["participant", "scene_order"]
).reset_index(drop=True)

print("Final scene-level rows:", len(scene_data))
print("Participants:", scene_data["participant"].nunique())
print("POMS remains separate because it is Pre/Post for the whole session, not scene-specific.")

display(
    scene_data[
        [
            "participant",
            "scene_code",
            "joint_retention_percent",
            "eeg_theta_relative_percent",
            "eeg_alpha_relative_percent",
            "eeg_beta_relative_percent",
            "ecg_mean_hr_bpm",
            "ecg_qc",
        ]
    ].head(9)
)

Final scene-level rows: 60
Participants: 20
POMS remains separate because it is Pre/Post for the whole session, not scene-specific.


,participant,scene_code,joint_retention_percent,eeg_theta_relative_percent,eeg_alpha_relative_percent,eeg_beta_relative_percent,ecg_mean_hr_bpm,ecg_qc
0,IC01,BG,73.913043,66.773248,15.699090,16.160337,76.472810,ok
1,IC01,G,97.183099,87.865620,7.805933,4.426262,78.788133,ok
2,IC01,GR,100.000000,75.154811,12.915692,11.661407,76.400679,ok
3,IC02,BG,95.652174,78.403848,12.441519,9.549123,64.066368,ok
4,IC02,G,100.000000,60.377917,20.803363,17.923323,67.563715,ok
5,IC02,GR,94.366197,59.594660,21.531544,17.738448,64.353763,ok
6,IC03,BG,97.183099,76.578888,13.571728,8.799285,91.560956,ok
7,IC03,G,98.591549,68.125645,20.042685,11.720368,91.465342,ok
8,IC03,GR,100.000000,75.974663,15.719867,8.410298,90.163474,ok


In [8]:
# =========================================================
# 7. EEG STATISTICS
# =========================================================

EEG_FEATURES = {
    "eeg_theta_relative_percent": "Relative theta power (%)",
    "eeg_alpha_relative_percent": "Relative alpha power (%)",
    "eeg_beta_relative_percent": "Relative beta power (%)",
}

eeg_omnibus_tables = []
eeg_pairwise_tables = []
eeg_descriptive_tables = []
eeg_pivots = {}

for feature in EEG_FEATURES:
    omnibus, pairwise, descriptive, pivot = run_three_scene_nonparametric(
        scene_data,
        feature,
    )

    omnibus["label"] = EEG_FEATURES[feature]
    pairwise["label"] = EEG_FEATURES[feature]
    descriptive["label"] = EEG_FEATURES[feature]

    eeg_omnibus_tables.append(omnibus)
    eeg_pairwise_tables.append(pairwise)
    eeg_descriptive_tables.append(descriptive)
    eeg_pivots[feature] = pivot

eeg_omnibus = pd.concat(eeg_omnibus_tables, ignore_index=True)
eeg_pairwise = pd.concat(eeg_pairwise_tables, ignore_index=True)
eeg_descriptive = pd.concat(eeg_descriptive_tables, ignore_index=True)

print("EEG omnibus tests:")
display(eeg_omnibus)

print("\nEEG pairwise tests (Holm-adjusted within each EEG feature):")
display(eeg_pairwise)

print("\nEEG descriptive statistics:")
display(eeg_descriptive)

EEG omnibus tests:


,feature,n_complete,friedman_chi2,df,p_value,kendalls_w,label
0,eeg_theta_relative_percent,20,3.9,2,0.142274,0.0975,Relative theta power (%)
1,eeg_alpha_relative_percent,20,1.3,2,0.522046,0.0325,Relative alpha power (%)
2,eeg_beta_relative_percent,20,5.2,2,0.074274,0.1300,Relative beta power (%)



EEG pairwise tests (Holm-adjusted within each EEG feature):


,feature,scene_a,scene_b,comparison,n_complete,wilcoxon_statistic,p_raw,rank_biserial_a_minus_b,median_difference_a_minus_b,p_holm,label
0,eeg_theta_relative_percent,BG,G,Blue-Green vs Green,20,55.0,0.063723,0.476190,3.927626,0.191168,Relative theta power (%)
1,eeg_theta_relative_percent,BG,GR,Blue-Green vs Grey,20,89.0,0.570597,0.152381,0.520618,0.570597,Relative theta power (%)
2,eeg_theta_relative_percent,G,GR,Green vs Grey,20,62.0,0.113987,-0.409524,-2.453922,0.227974,Relative theta power (%)
3,eeg_alpha_relative_percent,BG,G,Blue-Green vs Green,20,103.0,0.956329,-0.019048,-0.524360,1.000000,Relative alpha power (%)
4,eeg_alpha_relative_percent,BG,GR,Blue-Green vs Grey,20,96.0,0.756166,-0.085714,0.745344,1.000000,Relative alpha power (%)
5,eeg_alpha_relative_percent,G,GR,Green vs Grey,20,105.0,1.000000,0.000000,0.939168,1.000000,Relative alpha power (%)
6,eeg_beta_relative_percent,BG,G,Blue-Green vs Green,20,40.0,0.013617,-0.619048,-3.490759,0.040850,Relative beta power (%)
7,eeg_beta_relative_percent,BG,GR,Blue-Green vs Grey,20,88.0,0.545876,-0.161905,-0.696123,0.545876,Relative beta power (%)
8,eeg_beta_relative_percent,G,GR,Green vs Grey,20,51.0,0.044054,0.514286,1.153574,0.088108,Relative beta power (%)



EEG descriptive statistics:


,feature,scene_code,scene_name,n,median,q1,q3,iqr,label
0,eeg_theta_relative_percent,BG,Blue-Green,20,68.252828,57.320986,77.809612,20.488626,Relative theta power (%)
1,eeg_theta_relative_percent,G,Green,20,57.701244,52.057287,73.411491,21.354204,Relative theta power (%)
2,eeg_theta_relative_percent,GR,Grey,20,65.500776,55.335228,75.430282,20.095054,Relative theta power (%)
3,eeg_alpha_relative_percent,BG,Blue-Green,20,16.026320,14.434611,21.769884,7.335273,Relative alpha power (%)
4,eeg_alpha_relative_percent,G,Green,20,17.040027,15.456769,20.935484,5.478715,Relative alpha power (%)
5,eeg_alpha_relative_percent,GR,Grey,20,15.711657,14.177477,23.315965,9.138488,Relative alpha power (%)
6,eeg_beta_relative_percent,BG,Blue-Green,20,13.804805,8.757871,22.286125,13.528254,Relative beta power (%)
7,eeg_beta_relative_percent,G,Green,20,19.206645,10.385692,28.463520,18.077827,Relative beta power (%)
8,eeg_beta_relative_percent,GR,Grey,20,14.224557,10.018166,23.795501,13.777335,Relative beta power (%)


In [9]:
# =========================================================
# 8. ECG STATISTICS
# =========================================================

ecg_for_stats = scene_data.loc[
    scene_data["ecg_qc"] == "ok",
    ["participant", "scene_code", "ecg_mean_hr_bpm"],
].copy()

ecg_complete = (
    ecg_for_stats.pivot(
        index="participant",
        columns="scene_code",
        values="ecg_mean_hr_bpm",
    )
    .reindex(columns=SCENE_ORDER)
    .dropna()
)

ecg_complete_participants = ecg_complete.index.tolist()

print("ECG complete-case participants:", len(ecg_complete_participants))
print(ecg_complete_participants)

if len(ecg_complete_participants) >= 3:
    ecg_stats_input = ecg_for_stats.loc[
        ecg_for_stats["participant"].isin(ecg_complete_participants)
    ].copy()

    ecg_omnibus, ecg_pairwise, ecg_descriptive, ecg_pivot = (
        run_three_scene_nonparametric(
            ecg_stats_input,
            "ecg_mean_hr_bpm",
        )
    )

    ecg_omnibus["label"] = "Mean heart rate (bpm)"
    ecg_pairwise["label"] = "Mean heart rate (bpm)"
    ecg_descriptive["label"] = "Mean heart rate (bpm)"

    print("\nECG omnibus test:")
    display(ecg_omnibus)

    print("\nECG pairwise tests:")
    display(ecg_pairwise)

    print("\nECG descriptive statistics:")
    display(ecg_descriptive)

else:
    warnings.warn(
        "Fewer than 3 participants have complete ECG across all scenes. "
        "ECG inferential statistics were not run."
    )

    ecg_omnibus = pd.DataFrame()
    ecg_pairwise = pd.DataFrame()
    ecg_descriptive = pd.DataFrame()
    ecg_pivot = pd.DataFrame()

ECG complete-case participants: 20
['IC01', 'IC02', 'IC03', 'IC04', 'IC05', 'IC06', 'IC07', 'IC08', 'IC09', 'IC10', 'IC11', 'IC13', 'IC14', 'IC15', 'N01', 'N02', 'N03', 'N04', 'N05', 'N06']

ECG omnibus test:


,feature,n_complete,friedman_chi2,df,p_value,kendalls_w,label
0,ecg_mean_hr_bpm,20,4.9,2,0.086294,0.1225,Mean heart rate (bpm)



ECG pairwise tests:


,feature,scene_a,scene_b,comparison,n_complete,wilcoxon_statistic,p_raw,rank_biserial_a_minus_b,median_difference_a_minus_b,p_holm,label
0,ecg_mean_hr_bpm,BG,G,Blue-Green vs Green,20,77.0,0.311794,0.266667,0.762713,0.623589,Mean heart rate (bpm)
1,ecg_mean_hr_bpm,BG,GR,Blue-Green vs Grey,20,44.0,0.021484,0.580952,1.524431,0.064453,Mean heart rate (bpm)
2,ecg_mean_hr_bpm,G,GR,Green vs Grey,20,81.0,0.388376,0.228571,0.801178,0.623589,Mean heart rate (bpm)



ECG descriptive statistics:


,feature,scene_code,scene_name,n,median,q1,q3,iqr,label
0,ecg_mean_hr_bpm,BG,Blue-Green,20,69.080816,66.049211,75.986687,9.937477,Mean heart rate (bpm)
1,ecg_mean_hr_bpm,G,Green,20,68.476699,64.192231,76.698362,12.506130,Mean heart rate (bpm)
2,ecg_mean_hr_bpm,GR,Grey,20,67.260093,64.268018,75.677578,11.409560,Mean heart rate (bpm)


In [10]:
# =========================================================
# 9. POMS STATISTICS
# =========================================================

pre_tmd = poms_wide["TMD_Pre"].to_numpy(dtype=float)
post_tmd = poms_wide["TMD_Post"].to_numpy(dtype=float)

poms_test = stats.wilcoxon(
    post_tmd,
    pre_tmd,
    alternative="two-sided",
    zero_method="wilcox",
    method="auto",
)

poms_statistics = pd.DataFrame([{
    "measure": "TMD",
    "n": len(poms_wide),
    "pre_median": float(np.median(pre_tmd)),
    "pre_q1": float(np.percentile(pre_tmd, 25)),
    "pre_q3": float(np.percentile(pre_tmd, 75)),
    "post_median": float(np.median(post_tmd)),
    "post_q1": float(np.percentile(post_tmd, 25)),
    "post_q3": float(np.percentile(post_tmd, 75)),
    "delta_median_post_minus_pre": float(
        np.median(post_tmd - pre_tmd)
    ),
    "wilcoxon_statistic": float(poms_test.statistic),
    "p_value": float(poms_test.pvalue),
    "rank_biserial_post_minus_pre": rank_biserial_paired(
        post_tmd,
        pre_tmd,
    ),
}])

# Seven subscales are descriptive only.
poms_subscale_descriptive_rows = []

for measure in subscales:
    pre = poms_wide[f"{measure}_Pre"].to_numpy(dtype=float)
    post = poms_wide[f"{measure}_Post"].to_numpy(dtype=float)

    poms_subscale_descriptive_rows.append({
        "measure": measure,
        "n": len(pre),
        "pre_median": float(np.median(pre)),
        "pre_q1": float(np.percentile(pre, 25)),
        "pre_q3": float(np.percentile(pre, 75)),
        "post_median": float(np.median(post)),
        "post_q1": float(np.percentile(post, 25)),
        "post_q3": float(np.percentile(post, 75)),
        "delta_median_post_minus_pre": float(
            np.median(post - pre)
        ),
    })

poms_subscale_descriptive = pd.DataFrame(
    poms_subscale_descriptive_rows
)

print("Primary POMS result:")
display(poms_statistics)

print("\nPOMS subscales — descriptive only:")
display(poms_subscale_descriptive)

Primary POMS result:


,measure,n,pre_median,pre_q1,pre_q3,post_median,post_q1,post_q3,delta_median_post_minus_pre,wilcoxon_statistic,p_value,rank_biserial_post_minus_pre
0,TMD,20,93.5,86.5,99.0,86.5,80.0,92.5,-3.0,26.5,0.003339,-0.747619



POMS subscales — descriptive only:


,measure,n,pre_median,pre_q1,pre_q3,post_median,post_q1,post_q3,delta_median_post_minus_pre
0,TEN,20,3.0,1.00,4.25,1.0,0.00,2.25,-2.0
1,DEP,20,0.0,0.00,3.00,0.0,0.00,0.00,0.0
2,ANG,20,0.0,0.00,2.00,0.0,0.00,0.00,0.0
3,FAT,20,3.0,1.75,7.25,2.0,0.75,4.00,-1.5
4,CON,20,2.0,0.75,4.00,1.0,0.00,2.00,-1.0
5,VIG,20,5.0,3.00,9.00,6.0,2.00,9.00,-0.5
6,ERA,20,12.0,10.00,14.50,12.5,10.75,17.00,1.0


In [11]:
# =========================================================
# 10. PUBLICATION-READY FIGURES
# =========================================================

from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})

# -----------------------------
# Shared plotting style
# -----------------------------
grid_color = "0.88"

box_style = dict(
    boxprops=dict(color="black", linewidth=1.4),
    whiskerprops=dict(color="black", linewidth=1.2),
    capprops=dict(color="black", linewidth=1.2),
    medianprops=dict(color="#d95f02", linewidth=2.0),
)

def clean_axis(ax):
    ax.grid(axis="y", color=grid_color, linewidth=0.7)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)

# -----------------------------
# Stable participant colours
# -----------------------------
all_participants = set()

for feature in EEG_FEATURES:
    all_participants.update(eeg_pivots[feature].index.tolist())

if len(ecg_pivot) > 0:
    all_participants.update(ecg_pivot.index.tolist())

all_participants.update(
    poms_wide["Participant"].astype(str).str.strip().tolist()
)

# Standardise participant IDs as strings
all_participants = sorted(
    {str(pid).strip() for pid in all_participants},
    key=str
)

# Stable colour for each participant
cmap = plt.get_cmap("tab20")
participant_colors = {
    pid: cmap(i % 20)
    for i, pid in enumerate(all_participants)
}

legend_handle = Line2D(
    [0], [0],
    color="0.60",
    marker="o",
    linewidth=1.0,
    markersize=5,
    label="Individual participants"
)

# -----------------------------
# Title map for EEG bands
# -----------------------------
eeg_title_map = {
    "eeg_theta_relative_percent": "Theta (4–8 Hz)",
    "eeg_alpha_relative_percent": "Alpha (8–13 Hz)",
    "eeg_beta_relative_percent": "Beta (13–30 Hz)",
}

# =========================================================
# Figure 1: EEG
# =========================================================
fig, axes = plt.subplots(
    3, 1,
    figsize=(4.8, 8.6),
    constrained_layout=True
)

for ax, (feature, label) in zip(axes, EEG_FEATURES.items()):
    pivot = eeg_pivots[feature]

    # individual paired trajectories
    for pid, row in pivot.iterrows():
        ax.plot(
            [1, 2, 3],
            row[SCENE_ORDER].to_numpy(dtype=float),
            color="0.70",
            marker="o",
            markersize=3.0,
            linewidth=0.9,
            alpha=0.40,
            zorder=1,
        )

    # boxplots
    ax.boxplot(
        [pivot[scene].to_numpy(dtype=float) for scene in SCENE_ORDER],
        positions=[1, 2, 3],
        widths=0.42,
        showfliers=False,
        zorder=2,
        **box_style,
    )

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([SCENE_NAMES[s] for s in SCENE_ORDER])
    ax.set_ylabel("Relative power (%)")

    title_text = eeg_title_map[feature]
    ax.set_title(title_text, pad=2)

    clean_axis(ax)

# Put a simple explanatory legend inside the EEG figure
axes[2].legend(
    handles=[legend_handle],
    loc="upper right",
    frameon=False,
    fontsize=6
)

fig.suptitle(
    "EEG relative band power across 3 environmental scenarios",
    y=1.01
)

eeg_figure_paths = save_figure(
    fig,
    "Figure_S4_1_EEG_relative_band_power",
)

print("Saved EEG figure:", eeg_figure_paths)


# =========================================================
# Figure 2: ECG
# =========================================================
if len(ecg_pivot) >= 3:
    fig, ax = plt.subplots(figsize=(4.6, 3.5))

    for pid, row in ecg_pivot.iterrows():
        ax.plot(
            [1, 2, 3],
            row[SCENE_ORDER].to_numpy(dtype=float),
            color="0.70",
            marker="o",
            markersize=3.0,
            linewidth=0.9,
            alpha=0.40,
            zorder=1,
        )

    ax.boxplot(
        [ecg_pivot[scene].to_numpy(dtype=float) for scene in SCENE_ORDER],
        positions=[1, 2, 3],
        widths=0.42,
        showfliers=False,
        zorder=2,
        **box_style,
    )

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([SCENE_NAMES[s] for s in SCENE_ORDER])

    ax.set_ylabel("Mean heart rate (bpm)")
    ax.set_title("ECG mean heart rate across 3 environmental scenarios")

    clean_axis(ax)

    ax.legend(
        handles=[legend_handle],
        loc="upper right",
        frameon=False,
        fontsize=6
    )

    fig.tight_layout()

    ecg_figure_paths = save_figure(
        fig,
        "Figure_S4_2_ECG_mean_heart_rate",
    )

    print("Saved ECG figure:", ecg_figure_paths)


# =========================================================
# Figure 3: POMS
# =========================================================
fig, ax = plt.subplots(figsize=(4.3, 3.5))

for _, row in poms_wide.iterrows():
    pid = str(row["Participant"]).strip()

    ax.plot(
        [1, 2],
        [row["TMD_Pre"], row["TMD_Post"]],
        color="0.70",
        marker="o",
        markersize=3.0,
        linewidth=0.9,
        alpha=0.45,
        zorder=1,
    )

ax.boxplot(
    [
        poms_wide["TMD_Pre"].to_numpy(dtype=float),
        poms_wide["TMD_Post"].to_numpy(dtype=float),
    ],
    positions=[1, 2],
    widths=0.38,
    showfliers=False,
    zorder=2,
    **box_style,
)

ax.set_xticks([1, 2])
ax.set_xticklabels(["Pre", "Post"])

ax.set_ylabel("TMD score (+100)")
ax.set_title("POMS total mood disturbance")

clean_axis(ax)

ax.legend(
    handles=[legend_handle],
    loc="upper right",
    frameon=False,
    fontsize=6
)

poms_figure_paths = save_figure(
    fig,
    "Figure_S4_3_POMS_TMD_Pre_Post",
)

print("Saved POMS figure:", poms_figure_paths)

Saved EEG figure: (WindowsPath('C:/Users/Harvey McWalter/Desktop/EEG_Data_22/Script4_Final_Analysis/figures/Figure_S4_1_EEG_relative_band_power.png'), WindowsPath('C:/Users/Harvey McWalter/Desktop/EEG_Data_22/Script4_Final_Analysis/figures/Figure_S4_1_EEG_relative_band_power.pdf'))
Saved ECG figure: (WindowsPath('C:/Users/Harvey McWalter/Desktop/EEG_Data_22/Script4_Final_Analysis/figures/Figure_S4_2_ECG_mean_heart_rate.png'), WindowsPath('C:/Users/Harvey McWalter/Desktop/EEG_Data_22/Script4_Final_Analysis/figures/Figure_S4_2_ECG_mean_heart_rate.pdf'))
Saved POMS figure: (WindowsPath('C:/Users/Harvey McWalter/Desktop/EEG_Data_22/Script4_Final_Analysis/figures/Figure_S4_3_POMS_TMD_Pre_Post.png'), WindowsPath('C:/Users/Harvey McWalter/Desktop/EEG_Data_22/Script4_Final_Analysis/figures/Figure_S4_3_POMS_TMD_Pre_Post.pdf'))


In [12]:
# =========================================================
# 11. EXPORT FINAL TABLES
# =========================================================

scene_data_file = OUTPUT_DIR / "script4_scene_analysis_data.csv"
poms_file = OUTPUT_DIR / "script4_poms_change.csv"

eeg_descriptive_file = OUTPUT_DIR / "script4_eeg_descriptive.csv"
eeg_omnibus_file = OUTPUT_DIR / "script4_eeg_friedman.csv"
eeg_pairwise_file = OUTPUT_DIR / "script4_eeg_pairwise_wilcoxon.csv"

ecg_summary_file = OUTPUT_DIR / "script4_ecg_scene_summary.csv"
ecg_omnibus_file = OUTPUT_DIR / "script4_ecg_friedman.csv"
ecg_pairwise_file = OUTPUT_DIR / "script4_ecg_pairwise_wilcoxon.csv"

poms_statistics_file = OUTPUT_DIR / "script4_poms_statistics.csv"
poms_subscale_file = OUTPUT_DIR / "script4_poms_subscale_descriptive.csv"

excel_file = OUTPUT_DIR / "Script4_Final_Results.xlsx"

scene_data.to_csv(scene_data_file, index=False)
poms_wide.to_csv(poms_file, index=False)

eeg_descriptive.to_csv(eeg_descriptive_file, index=False)
eeg_omnibus.to_csv(eeg_omnibus_file, index=False)
eeg_pairwise.to_csv(eeg_pairwise_file, index=False)

ecg.to_csv(ecg_summary_file, index=False)

if not ecg_omnibus.empty:
    ecg_omnibus.to_csv(ecg_omnibus_file, index=False)
    ecg_pairwise.to_csv(ecg_pairwise_file, index=False)

poms_statistics.to_csv(poms_statistics_file, index=False)
poms_subscale_descriptive.to_csv(poms_subscale_file, index=False)

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    scene_data.to_excel(writer, sheet_name="Scene data", index=False)
    eeg_descriptive.to_excel(writer, sheet_name="EEG descriptive", index=False)
    eeg_omnibus.to_excel(writer, sheet_name="EEG Friedman", index=False)
    eeg_pairwise.to_excel(writer, sheet_name="EEG pairwise", index=False)

    ecg.to_excel(writer, sheet_name="ECG extraction", index=False)

    if not ecg_descriptive.empty:
        ecg_descriptive.to_excel(writer, sheet_name="ECG descriptive", index=False)
        ecg_omnibus.to_excel(writer, sheet_name="ECG Friedman", index=False)
        ecg_pairwise.to_excel(writer, sheet_name="ECG pairwise", index=False)

    poms_wide.to_excel(writer, sheet_name="POMS participant", index=False)
    poms_statistics.to_excel(writer, sheet_name="POMS statistics", index=False)
    poms_subscale_descriptive.to_excel(
        writer,
        sheet_name="POMS subscales",
        index=False,
    )

print("\n=========================================================")
print("SCRIPT 4 COMPLETE")
print("=========================================================")
print("Final EEG participants:", scene_data["participant"].nunique())
print("Final EEG scene rows:", len(scene_data))
print("POMS paired participants:", len(poms_wide))
print("ECG complete-case participants:", len(ecg_complete_participants))
print("Excluded throughout:", sorted(EXCLUDED_PARTICIPANTS))
print("\nResults folder:", OUTPUT_DIR)
print("Final Excel workbook:", excel_file)
print("Figures folder:", FIGURE_DIR)


SCRIPT 4 COMPLETE
Final EEG participants: 20
Final EEG scene rows: 60
POMS paired participants: 20
ECG complete-case participants: 20
Excluded throughout: ['IC12', 'N07']

Results folder: C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script4_Final_Analysis
Final Excel workbook: C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script4_Final_Analysis\Script4_Final_Results.xlsx
Figures folder: C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script4_Final_Analysis\figures


## 12. Trial-position / order sensitivity analysis (added)

The original Friedman, Wilcoxon, Holm-corrected pairwise analyses, effect sizes, and all existing figure-generation code above are left unchanged.

As a secondary sensitivity analysis, the pre-specified counterbalanced exposure sequence is reconstructed for each participant and the actual trial position (1–3) is added only to a separate analysis copy of the scene-level dataset. For each EEG relative-power outcome and mean heart rate, a random-intercept linear mixed-effects model is fitted with **environmental scene** and **trial position** as categorical fixed effects and **participant** as a random intercept:

`outcome ~ scene + trial_position + (1 | participant)`

This provides an order-adjusted check of the primary scene comparisons without replacing the rank-based primary analysis. The approach follows standard mixed-effects treatment of repeated observations (Baayen, Davidson & Bates, 2008) and the repeated-participant mixed-model logic used in environmental cardiovascular field research (Lanki et al., 2017). POMS is not included in this sensitivity analysis because it was measured only before and after the complete three-scene session, not at each trial position.


In [13]:

# =========================================================
# 12. ADDITION: TRIAL-POSITION / ORDER SENSITIVITY ANALYSIS
# =========================================================
#
# IMPORTANT:
# - The original Friedman / Wilcoxon / Holm primary analyses above are unchanged.
# - This is a secondary sensitivity analysis only.
# - "scene_order" created earlier is only a display/sorting variable and is NOT
#   the participant's actual trial position.
# - Actual trial position is reconstructed from the pre-specified
#   counterbalanced exposure sequences used in the field protocol.
#
# Sensitivity model:
#   outcome ~ scene + trial_position + (1 | participant)
#
# Scene and trial position are categorical fixed effects.
# Participant is a random intercept to account for repeated observations.
#
# Methodological rationale:
# - Mixed-effects models are widely used for repeated-measures data because they
#   explicitly account for within-participant dependence (Baayen et al., 2008).
# - Period/order effects can be included as fixed effects in repeated/crossover
#   analyses, while participant is modelled as a random effect.
# - In an environmental field study, Lanki et al. (2017) similarly used mixed
#   models with participants treated as repeated factors for cardiovascular
#   outcomes across environmental visits.
#
# References:
# Baayen, R. H., Davidson, D. J., & Bates, D. M. (2008).
# Mixed-effects modeling with crossed random effects for subjects and items.
# Journal of Memory and Language, 59(4), 390-412.
#
# Lanki, T. et al. (2017).
# Acute effects of visits to urban green environments on cardiovascular
# physiology in women: A field experiment.
# Environmental Research, 159, 176-185.

try:
    import statsmodels
    import statsmodels.formula.api as smf
except ImportError as error:
    raise ImportError(
        "The trial-position sensitivity analysis requires statsmodels. "
        "Install statsmodels in the same Python environment, then rerun this cell."
    ) from error


# ---------------------------------------------------------
# 12.1 Reconstruct the pre-specified exposure order
# ---------------------------------------------------------

TRIAL_SEQUENCE_DEFINITIONS = {
    "A": {
        "order": ["G", "GR", "BG"],
        "participants": ["IC01", "N01", "IC05", "N05", "IC12", "N07"],
    },
    "B": {
        "order": ["GR", "G", "BG"],
        "participants": ["N02", "IC03", "IC07", "IC09", "IC11", "IC14"],
    },
    "C": {
        "order": ["BG", "G", "GR"],
        "participants": ["IC02", "N03", "IC06", "IC10", "IC13"],
    },
    "D": {
        "order": ["BG", "GR", "G"],
        "participants": ["IC04", "N04", "IC08", "N06", "IC15"],
    },
}

sequence_members = [
    participant
    for definition in TRIAL_SEQUENCE_DEFINITIONS.values()
    for participant in definition["participants"]
]

if len(sequence_members) != 22 or len(set(sequence_members)) != 22:
    raise ValueError(
        "Trial sequence definitions must contain each of the 22 participants exactly once."
    )

if set(sequence_members) != set(EXPECTED_PARTICIPANTS):
    raise ValueError(
        "Trial sequence participant IDs do not match EXPECTED_PARTICIPANTS."
    )

trial_order_rows = []

for sequence_group, definition in TRIAL_SEQUENCE_DEFINITIONS.items():
    for participant in definition["participants"]:
        if participant not in INCLUDED_PARTICIPANTS:
            continue

        for trial_position, scene_code in enumerate(definition["order"], start=1):
            trial_order_rows.append({
                "participant": participant,
                "sequence_group": sequence_group,
                "trial_position": trial_position,
                "scene_code": scene_code,
            })

trial_order_lookup = pd.DataFrame(trial_order_rows)

if (
    len(trial_order_lookup) != 60
    or trial_order_lookup["participant"].nunique() != 20
):
    raise ValueError(
        "Expected 60 participant-scene trial-order rows from 20 included participants."
    )

if trial_order_lookup.duplicated(["participant", "scene_code"]).any():
    raise ValueError("Duplicate participant-scene keys in trial-order lookup.")

order_sensitivity_data = scene_data.merge(
    trial_order_lookup,
    on=["participant", "scene_code"],
    how="left",
    validate="one_to_one",
)

if order_sensitivity_data[
    ["sequence_group", "trial_position"]
].isna().any().any():
    raise ValueError("Missing sequence/trial-position information after merge.")

print("Trial-order lookup validated.")
print("\nScene counts by actual trial position:")
display(
    pd.crosstab(
        order_sensitivity_data["trial_position"],
        order_sensitivity_data["scene_code"],
    ).reindex(index=[1, 2, 3], columns=SCENE_ORDER, fill_value=0)
)


# ---------------------------------------------------------
# 12.2 Mixed-model helper
# ---------------------------------------------------------

SENSITIVITY_FEATURES = {
    "eeg_theta_relative_percent": "Relative theta power (%)",
    "eeg_alpha_relative_percent": "Relative alpha power (%)",
    "eeg_beta_relative_percent": "Relative beta power (%)",
    "ecg_mean_hr_bpm": "Mean heart rate (bpm)",
}

SENSITIVITY_SCENE_PAIRS = [
    ("BG", "G"),
    ("BG", "GR"),
    ("G", "GR"),
]


def fit_order_adjusted_mixed_model(table, value_column, label):
    model_data = table[
        ["participant", "scene_code", "trial_position", value_column]
    ].copy()

    model_data[value_column] = pd.to_numeric(
        model_data[value_column],
        errors="coerce",
    )
    model_data = model_data.dropna(subset=[value_column]).copy()

    if model_data.empty:
        raise ValueError(f"{value_column}: no valid observations.")

    # Use only participants contributing all three scenes so the sensitivity
    # analysis remains directly comparable with the primary paired analysis.
    counts = model_data.groupby("participant")["scene_code"].nunique()
    complete_participants = counts[counts == 3].index.tolist()

    model_data = model_data.loc[
        model_data["participant"].isin(complete_participants)
    ].copy()

    if len(complete_participants) < 3:
        raise ValueError(
            f"{value_column}: fewer than 3 complete participants."
        )

    model_data["scene_code"] = pd.Categorical(
        model_data["scene_code"],
        categories=SCENE_ORDER,
        ordered=True,
    )
    model_data["trial_position"] = pd.Categorical(
        model_data["trial_position"].astype(int),
        categories=[1, 2, 3],
        ordered=True,
    )

    formula = f"{value_column} ~ C(scene_code) + C(trial_position)"

    model = smf.mixedlm(
        formula,
        data=model_data,
        groups=model_data["participant"],
        re_formula="1",
    )

    fit_result = None
    optimizer_used = None
    fit_messages = []

    # Small random-intercept models can occasionally be sensitive to the
    # numerical optimiser. Try a short, pre-specified fallback sequence.
    for optimizer in ["lbfgs", "powell", "bfgs"]:
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                candidate = model.fit(
                    reml=True,
                    method=optimizer,
                    maxiter=2000,
                    disp=False,
                )

            warning_text = " | ".join(
                str(item.message) for item in caught
            )
            fit_messages.append(
                f"{optimizer}: converged={bool(candidate.converged)}"
                + (f"; {warning_text}" if warning_text else "")
            )

            if candidate.converged:
                fit_result = candidate
                optimizer_used = optimizer
                break

        except Exception as error:
            fit_messages.append(
                f"{optimizer}: ERROR {type(error).__name__}: {error}"
            )

    if fit_result is None:
        raise RuntimeError(
            f"{value_column}: mixed model failed to converge. "
            + " || ".join(fit_messages)
        )

    wald_terms = fit_result.wald_test_terms(scalar=True).table

    scene_row = wald_terms.loc["C(scene_code)"]
    position_row = wald_terms.loc["C(trial_position)"]

    omnibus_row = {
        "feature": value_column,
        "label": label,
        "n_participants": len(complete_participants),
        "n_observations": len(model_data),
        "model": "Random-intercept LMM: outcome ~ scene + trial_position + (1|participant)",
        "reml": True,
        "optimizer": optimizer_used,
        "converged": bool(fit_result.converged),
        "scene_wald_chi2": float(scene_row["statistic"]),
        "scene_df": int(scene_row["df_constraint"]),
        "scene_p_value": float(scene_row["pvalue"]),
        "trial_position_wald_chi2": float(position_row["statistic"]),
        "trial_position_df": int(position_row["df_constraint"]),
        "trial_position_p_value": float(position_row["pvalue"]),
        "participant_random_intercept_variance": float(
            fit_result.cov_re.iloc[0, 0]
        ),
        "residual_variance": float(fit_result.scale),
        "fit_messages": " || ".join(fit_messages),
    }

    fe_names = fit_result.fe_params.index.tolist()
    ci = fit_result.conf_int().loc[fe_names]

    fixed_rows = []
    for term in fe_names:
        fixed_rows.append({
            "feature": value_column,
            "label": label,
            "term": term,
            "estimate": float(fit_result.fe_params[term]),
            "standard_error": float(fit_result.bse_fe[term]),
            "ci95_low": float(ci.loc[term, 0]),
            "ci95_high": float(ci.loc[term, 1]),
            "p_value": float(fit_result.pvalues[term]),
        })

    # Adjusted scene contrasts. Trial-position coefficients cancel from
    # scene-vs-scene contrasts, while the model estimates them simultaneously.
    fe = fit_result.fe_params
    cov_fe = fit_result.cov_params().loc[fe_names, fe_names]

    def scene_design_vector(scene_code):
        vector = pd.Series(0.0, index=fe_names)
        vector["Intercept"] = 1.0

        if scene_code != "BG":
            term = f"C(scene_code)[T.{scene_code}]"
            if term not in vector.index:
                raise KeyError(
                    f"Expected fixed-effect term '{term}' not found."
                )
            vector[term] = 1.0

        return vector.to_numpy(dtype=float)

    contrast_rows = []
    raw_p_values = []

    for scene_a, scene_b in SENSITIVITY_SCENE_PAIRS:
        contrast = (
            scene_design_vector(scene_a)
            - scene_design_vector(scene_b)
        )

        estimate = float(contrast @ fe.to_numpy(dtype=float))
        variance = float(
            contrast
            @ cov_fe.to_numpy(dtype=float)
            @ contrast
        )
        standard_error = float(np.sqrt(max(variance, 0.0)))

        if standard_error > 0:
            z_value = estimate / standard_error
            p_value = float(
                2.0 * stats.norm.sf(abs(z_value))
            )
            ci_low = estimate - 1.96 * standard_error
            ci_high = estimate + 1.96 * standard_error
        else:
            z_value = np.nan
            p_value = 1.0
            ci_low = estimate
            ci_high = estimate

        raw_p_values.append(p_value)

        contrast_rows.append({
            "feature": value_column,
            "label": label,
            "scene_a": scene_a,
            "scene_b": scene_b,
            "comparison": f"{SCENE_NAMES[scene_a]} vs {SCENE_NAMES[scene_b]}",
            "estimate_a_minus_b": estimate,
            "standard_error": standard_error,
            "ci95_low": float(ci_low),
            "ci95_high": float(ci_high),
            "z_value": float(z_value) if np.isfinite(z_value) else np.nan,
            "p_raw": p_value,
        })

    adjusted = holm_adjust(raw_p_values)

    for row, p_adj in zip(contrast_rows, adjusted):
        row["p_holm"] = float(p_adj)

    return (
        pd.DataFrame([omnibus_row]),
        pd.DataFrame(fixed_rows),
        pd.DataFrame(contrast_rows),
        model_data,
    )


# ---------------------------------------------------------
# 12.3 Run sensitivity models
# ---------------------------------------------------------

order_omnibus_tables = []
order_fixed_tables = []
order_contrast_tables = []
order_model_inputs = {}

for feature in EEG_FEATURES:
    omnibus, fixed, contrasts, model_input = (
        fit_order_adjusted_mixed_model(
            order_sensitivity_data,
            feature,
            EEG_FEATURES[feature],
        )
    )

    order_omnibus_tables.append(omnibus)
    order_fixed_tables.append(fixed)
    order_contrast_tables.append(contrasts)
    order_model_inputs[feature] = model_input


# Keep ECG sensitivity analysis aligned with the complete-case ECG cohort
# used in the primary Friedman/Wilcoxon analysis.
if len(ecg_complete_participants) >= 3:
    ecg_order_input = order_sensitivity_data.loc[
        order_sensitivity_data["participant"].isin(
            ecg_complete_participants
        )
        & (order_sensitivity_data["ecg_qc"] == "ok")
    ].copy()

    omnibus, fixed, contrasts, model_input = (
        fit_order_adjusted_mixed_model(
            ecg_order_input,
            "ecg_mean_hr_bpm",
            "Mean heart rate (bpm)",
        )
    )

    order_omnibus_tables.append(omnibus)
    order_fixed_tables.append(fixed)
    order_contrast_tables.append(contrasts)
    order_model_inputs["ecg_mean_hr_bpm"] = model_input


order_sensitivity_omnibus = pd.concat(
    order_omnibus_tables,
    ignore_index=True,
)
order_sensitivity_fixed_effects = pd.concat(
    order_fixed_tables,
    ignore_index=True,
)
order_sensitivity_scene_contrasts = pd.concat(
    order_contrast_tables,
    ignore_index=True,
)

print("\nOrder-adjusted mixed-model omnibus sensitivity results:")
display(order_sensitivity_omnibus)

print("\nOrder-adjusted scene contrasts (Holm-adjusted within each outcome):")
display(order_sensitivity_scene_contrasts)

print("\nFixed-effect coefficients:")
display(order_sensitivity_fixed_effects)


# ---------------------------------------------------------
# 12.4 Export sensitivity-analysis outputs
# ---------------------------------------------------------

trial_order_file = OUTPUT_DIR / "script4_trial_order_lookup.csv"
order_omnibus_file = OUTPUT_DIR / "script4_order_sensitivity_omnibus.csv"
order_fixed_file = OUTPUT_DIR / "script4_order_sensitivity_fixed_effects.csv"
order_contrasts_file = OUTPUT_DIR / "script4_order_sensitivity_scene_contrasts.csv"

trial_order_lookup.to_csv(trial_order_file, index=False)
order_sensitivity_omnibus.to_csv(order_omnibus_file, index=False)
order_sensitivity_fixed_effects.to_csv(order_fixed_file, index=False)
order_sensitivity_scene_contrasts.to_csv(
    order_contrasts_file,
    index=False,
)

# Append sensitivity tables to the existing Script 4 workbook generated above.
with pd.ExcelWriter(
    excel_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
) as writer:
    trial_order_lookup.to_excel(
        writer,
        sheet_name="Trial order lookup",
        index=False,
    )
    order_sensitivity_omnibus.to_excel(
        writer,
        sheet_name="Order sensitivity",
        index=False,
    )
    order_sensitivity_scene_contrasts.to_excel(
        writer,
        sheet_name="Order scene contrasts",
        index=False,
    )
    order_sensitivity_fixed_effects.to_excel(
        writer,
        sheet_name="Order fixed effects",
        index=False,
    )

print("\n=========================================================")
print("TRIAL-POSITION / ORDER SENSITIVITY EXTENSION COMPLETE")
print("=========================================================")
print("statsmodels version:", statsmodels.__version__)
print("Sensitivity CSV files saved in:", OUTPUT_DIR)
print("Sensitivity sheets appended to:", excel_file)


Trial-order lookup validated.

Scene counts by actual trial position:


scene_code,BG,G,GR
trial_position,,,
1,10,4,6
2,0,11,9
3,10,5,5



Order-adjusted mixed-model omnibus sensitivity results:


,feature,label,n_participants,n_observations,model,reml,optimizer,converged,scene_wald_chi2,scene_df,scene_p_value,trial_position_wald_chi2,trial_position_df,trial_position_p_value,participant_random_intercept_variance,residual_variance,fit_messages
0,eeg_theta_relative_percent,Relative theta power (%),20,60,Random-intercept LMM: outcome ~ scene + trial_...,True,lbfgs,True,3.521646,2,0.171903,37.558778,2,6.985783e-09,155.055161,26.352449,lbfgs: converged=True
1,eeg_alpha_relative_percent,Relative alpha power (%),20,60,Random-intercept LMM: outcome ~ scene + trial_...,True,lbfgs,True,0.699368,2,0.704911,14.455377,2,7.261976e-04,25.517081,5.962946,lbfgs: converged=True
2,eeg_beta_relative_percent,Relative beta power (%),20,60,Random-intercept LMM: outcome ~ scene + trial_...,True,lbfgs,True,7.142871,2,0.028115,25.370600,2,3.096309e-06,62.364416,17.601878,lbfgs: converged=True
3,ecg_mean_hr_bpm,Mean heart rate (bpm),20,60,Random-intercept LMM: outcome ~ scene + trial_...,True,lbfgs,True,5.682030,2,0.058366,7.057008,2,2.934880e-02,84.777318,4.310837,lbfgs: converged=True



Order-adjusted scene contrasts (Holm-adjusted within each outcome):


,feature,label,scene_a,scene_b,comparison,estimate_a_minus_b,standard_error,ci95_low,ci95_high,z_value,p_raw,p_holm
0,eeg_theta_relative_percent,Relative theta power (%),BG,G,Blue-Green vs Green,3.055910,1.855397,-0.580668,6.692487,1.647038,0.099550,0.298650
1,eeg_theta_relative_percent,Relative theta power (%),BG,GR,Blue-Green vs Grey,0.467520,1.782032,-3.025262,3.960302,0.262352,0.793050,0.793050
2,eeg_theta_relative_percent,Relative theta power (%),G,GR,Green vs Grey,-2.588389,1.633586,-5.790217,0.613439,-1.584483,0.113084,0.298650
3,eeg_alpha_relative_percent,Relative alpha power (%),BG,G,Blue-Green vs Green,0.627719,0.882585,-1.102148,2.357587,0.711228,0.476943,1.000000
4,eeg_alpha_relative_percent,Relative alpha power (%),BG,GR,Blue-Green vs Grey,0.060997,0.847687,-1.600469,1.722463,0.071956,0.942637,1.000000
5,eeg_alpha_relative_percent,Relative alpha power (%),G,GR,Green vs Grey,-0.566723,0.777073,-2.089786,0.956341,-0.729304,0.465816,1.000000
6,eeg_beta_relative_percent,Relative beta power (%),BG,G,Blue-Green vs Green,-3.271031,1.516372,-6.243120,-0.298942,-2.157143,0.030995,0.061989
7,eeg_beta_relative_percent,Relative beta power (%),BG,GR,Blue-Green vs Grey,-0.033301,1.456413,-2.887869,2.821268,-0.022865,0.981758,0.981758
8,eeg_beta_relative_percent,Relative beta power (%),G,GR,Green vs Grey,3.237730,1.335091,0.620951,5.854509,2.425100,0.015304,0.045912
9,ecg_mean_hr_bpm,Mean heart rate (bpm),BG,G,Blue-Green vs Green,1.136666,0.750425,-0.334166,2.607499,1.514698,0.129849,0.259698



Fixed-effect coefficients:


,feature,label,term,estimate,standard_error,ci95_low,ci95_high,p_value
0,eeg_theta_relative_percent,Relative theta power (%),Intercept,71.297178,3.119433,65.183203,77.411154,1.279171e-115
1,eeg_theta_relative_percent,Relative theta power (%),C(scene_code)[T.G],-3.055910,1.855397,-6.692421,0.580601,9.955016e-02
2,eeg_theta_relative_percent,Relative theta power (%),C(scene_code)[T.GR],-0.467520,1.782032,-3.960238,3.025198,7.930497e-01
3,eeg_theta_relative_percent,Relative theta power (%),C(trial_position)[T.2],-7.605657,1.826404,-11.185344,-4.025971,3.123381e-05
4,eeg_theta_relative_percent,Relative theta power (%),C(trial_position)[T.3],-9.570860,1.625397,-12.756580,-6.385139,3.901410e-09
5,eeg_alpha_relative_percent,Relative alpha power (%),Intercept,16.572039,1.312803,13.998992,19.145086,1.569006e-36
6,eeg_alpha_relative_percent,Relative alpha power (%),C(scene_code)[T.G],-0.627719,0.882585,-2.357555,1.102117,4.769432e-01
7,eeg_alpha_relative_percent,Relative alpha power (%),C(scene_code)[T.GR],-0.060997,0.847687,-1.722432,1.600439,9.426365e-01
8,eeg_alpha_relative_percent,Relative alpha power (%),C(trial_position)[T.2],2.630069,0.868794,0.927264,4.332875,2.467782e-03
9,eeg_alpha_relative_percent,Relative alpha power (%),C(trial_position)[T.3],2.640744,0.773178,1.125343,4.156145,6.367880e-04



TRIAL-POSITION / ORDER SENSITIVITY EXTENSION COMPLETE
statsmodels version: 0.14.4
Sensitivity CSV files saved in: C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script4_Final_Analysis
Sensitivity sheets appended to: C:\Users\Harvey McWalter\Desktop\EEG_Data_22\Script4_Final_Analysis\Script4_Final_Results.xlsx
